In [1]:
# 检查 Notebook 实际使用的 Python 和 PyTorch
import sys
import torch

print("Python 路径：", sys.executable)
print("PyTorch 版本：", torch.__version__)

Python 路径： C:\Users\OS11\Miniforge3\envs\torch-class\python.exe
PyTorch 版本： 2.7.1+cpu


In [2]:
# 输入和目标是固定数据，不需要求梯度
x = torch.tensor(1.0)
y = torch.tensor(4.0)

# 使用浮点数；requires_grad=True 让 PyTorch 跟踪与 w 有关的计算
w = torch.tensor(1.0, requires_grad=True)
y_pred = x * w ** 2
loss = torch.abs(y - y_pred)

print("当前 w：", w.item())
print("预测值：", y_pred.item())
print("损失：", loss.item())

当前 w： 1.0
预测值： 1.0
损失： 3.0


In [3]:
# 本单元格每次都会重新创建 w，可以独立重复运行
w = torch.tensor(1.0, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)

optimizer.zero_grad()
loss = torch.abs(y - x * w ** 2)
loss.backward()
print("更新前的 w：", w.item())
print("更新前的损失：", loss.item())
print("梯度：", w.grad.item())

optimizer.step()
# 参数更新后重新计算损失，让打印的 w 和损失属于同一时刻
with torch.no_grad():
    new_loss = torch.abs(y - x * w ** 2)
print("更新后的 w：", w.item())
print("更新后的损失：", new_loss.item())

更新前的 w： 1.0
更新前的损失： 3.0
梯度： -2.0
更新后的 w： 1.2000000476837158
更新后的损失： 2.559999942779541


In [4]:
def train_w(initial_w=1.0, epochs=1000, base_lr=0.1, decay=True):
    # 每次调用都从指定初值开始，避免上次实验的参数影响本次结果
    w = torch.tensor(float(initial_w), requires_grad=True)
    optimizer = torch.optim.SGD([w], lr=base_lr)
    history = []

    for epoch in range(1, epochs + 1):
        lr = base_lr / epoch if decay else base_lr
        optimizer.param_groups[0]["lr"] = lr
        optimizer.zero_grad()
        y_pred = x * w ** 2
        loss = torch.abs(y - y_pred)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            prediction = (x * w ** 2).item()
            current_loss = torch.abs(y - x * w ** 2).item()
        history.append((epoch, lr, w.item(), prediction, current_loss))
    return w.detach().item(), history

learned_w, history = train_w()
print("轮数    学习率          w           预测值        损失")
for index in [0, 1, 4, 9, 99, 999]:
    epoch, lr, value, prediction, current_loss = history[index]
    print(f"{epoch:4d}  {lr:.6f}  {value:11.6f}  {prediction:11.6f}  {current_loss:.6f}")
print("最终学到的 w：", learned_w)

轮数    学习率          w           预测值        损失
   1  0.100000     1.200000     1.440000  2.560000
   2  0.050000     1.320000     1.742400  2.257600
   5  0.020000     1.537536     2.364017  1.635983
  10  0.010000     1.746504     3.050276  0.949724
 100  0.001000     1.999979     3.999917  0.000083
1000  0.000100     2.000000     4.000000  0.000000
最终学到的 w： 2.0


In [5]:
for initial_w in [1.0, -1.0, 0.0]:
    final_w, records = train_w(initial_w=initial_w)
    print(f"初始 w={initial_w:5.1f} → 最终 w={final_w: .6f}，最终损失={records[-1][-1]:.6f}")

初始 w=  1.0 → 最终 w= 2.000000，最终损失=0.000000
初始 w= -1.0 → 最终 w=-2.000000，最终损失=0.000000
初始 w=  0.0 → 最终 w= 0.000000，最终损失=4.000000


In [6]:
_, fixed_history = train_w(epochs=40, base_lr=0.1, decay=False)
print("轮数       w           损失")
for epoch, lr, value, prediction, current_loss in fixed_history[-8:]:
    print(f"{epoch:4d}  {value:11.6f}  {current_loss:.6f}")

轮数       w           损失
  33     2.107632  0.442114
  34     1.686106  1.157047
  35     2.023327  0.093853
  36     1.618662  1.379935
  37     1.942394  0.227105
  38     2.330873  1.432968
  39     1.864698  0.522901
  40     2.237638  1.007023


In [7]:
# 修改这里的参数，完成自己的实验
my_initial_w = 0.5
my_base_lr = 0.1
my_w, my_history = train_w(initial_w=my_initial_w, base_lr=my_base_lr)
print("初始值：", my_initial_w)
print("最终 w：", my_w)
print("最终预测：", my_history[-1][-2])
print("最终损失：", my_history[-1][-1])

初始值： 0.5
最终 w： 2.000117301940918
最终预测： 4.000469207763672
最终损失： 0.000469207763671875


In [8]:
# 修改这里的参数，完成自己的实验
my_initial_w = -0.5
my_base_lr = 0.1
my_w, my_history = train_w(initial_w=my_initial_w, base_lr=my_base_lr)
print("初始值：", my_initial_w)
print("最终 w：", my_w)
print("最终预测：", my_history[-1][-2])
print("最终损失：", my_history[-1][-1])

初始值： -0.5
最终 w： -2.000117301940918
最终预测： 4.000469207763672
最终损失： 0.000469207763671875


In [9]:
_, fixed_history = train_w(epochs=40, base_lr=0.01, decay=False)
print("轮数       w           损失")
for epoch, lr, value, prediction, current_loss in fixed_history[-8:]:
    print(f"{epoch:4d}  {value:11.6f}  {current_loss:.6f}")

轮数       w           损失
  33     1.922231  0.305027
  34     1.960676  0.155750
  35     1.999889  0.000442
  36     2.039887  0.161140
  37     1.999089  0.003641
  38     2.039071  0.157812
  39     1.998290  0.006837
  40     2.038256  0.154486


In [10]:
# 输入和目标是固定数据，不需要求梯度
x = torch.tensor(1.0)
y = torch.tensor(9.0)

# 使用浮点数；requires_grad=True 让 PyTorch 跟踪与 w 有关的计算
w = torch.tensor(1.0, requires_grad=True)
y_pred = x * w ** 2
loss = torch.abs(y - y_pred)

print("当前 w：", w.item())
print("预测值：", y_pred.item())
print("损失：", loss.item())

当前 w： 1.0
预测值： 1.0
损失： 8.0


1.各个变量的含义：x:固定数据  y:固定数据  y_pred:预测的数据  loss:不同值的误差
2.0.5和-0.5的w都接近2，最终预测都接近4
3.随着轮数的增加，w和损失呈波动变化;相同轮数的条件下，0.01的w和损失都比0.1的小，但几乎没变，所以学习率更小不一定更好
4.初始化就不动了;w=0不动了，w=-2正确可能是因为算出来的预测值是4
5.y变成9.0后，w**2=9，理论上有两个解;y变成9.0后，当前w: 1.0,预测值: 1.0,损失: 8.0